# Asking the graph

Two engines read the same graph, and they are good at opposite things.

**AQLizer** is Arango's natural-language-to-AQL service. It writes a query, runs it,
and hands back both the rows and the query -- so an answer can be checked rather
than trusted. It is the one to use when the question has a shape: count, sum, rank,
traverse, or *find the ones that are missing something*.

**GraphRAG** is the retriever. It searches the entity descriptions and the source
text, follows the relations it lands on, and writes an answer from what it read.
It is the one to use when the question has no shape -- when it is phrased in words
the model does not use, or when the answer is spread across a dozen files.

Neither is a fallback for the other. The last section asks one question both ways
to show where the line is.

In [1]:
import logging
from sysml import nl

logging.disable(logging.INFO)  # both services narrate every step

---

# Part 1 -- AQLizer

Nothing below is a hand-written query. `sysml/aql_examples.md` teaches the model how
SysML concepts are laid out here -- an `attributes` map, an `owns`/`typedby` tree, a
`stated` flag on the edges -- and the AQL in every answer is what it wrote from that.

## 1. A mass budget

This is the hardest shape in the set: walk up to six hops down the containment tree
through two different edge types, pull two attributes off each element it lands on,
add them, sort by the sum, and cite where each number is declared.

In [2]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

Q  For each Saturn V stage, give its dry mass, its propellant mass and the sum of the two, sorted by the total, with the file and line each is declared on.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR saturnv IN sysml_Entities
     FILTER saturnv.entity_name == "SATURNV"
     FOR stage, edge IN 1..6 OUTBOUND saturnv sysml_Relations
       FILTER edge.relationship_type IN ["owns", "typedby"]
       FILTER stage.attributes.dryMass.value != null
       FILTER stage.attributes.propellantMass.value != null
       LET dryMass = stage.attributes.dryMass
       LET propellantMass = stage.attributes.propellantMass
       LET totalMass = dryMass.value + propellantMass.value
       SORT totalMass DESC
       RETURN {
         stage: stage.entity_name,
         dryMass: dryMass.value,
         dryMassUnit: dryMass.unit,
         propellantMass: propellantMass.value,
         propellantMassUnit: propellantMass.unit,
         totalMass: totalMass,
         at: CONCAT(stage.source_file, ":", s

Every figure is a number a file states, and the `file:line` beside it is where. That
is the half of the graph the lexer wrote; an LLM reading the same text reports the
masses as prose and cannot be summed.

## 2. What is *not* there

Coverage questions are the ones a requirements engineer actually asks, and they are
an anti-join: requirements with no incoming `satisfies` edge. Retrieval cannot answer
this at all -- there is no passage describing the absence of a relation.

In [3]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

Q  Which ten Apollo requirements have the most elements satisfying them, and how many Apollo requirements have none at all?

AQL
   WITH sysml_Entities, sysml_Relations
   
   LET satisfied_counts = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       LET satisfiers = (
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
           RETURN 1
       )
       RETURN {requirement: e.entity_name, count: LENGTH(satisfiers)}
   )
   
   LET unsatisfied_requirements = (
     FOR entry IN satisfied_counts
       FILTER entry.count == 0
       RETURN entry.requirement
   )
   
   LET top_ten_satisfied = (
     FOR entry IN satisfied_counts
       FILTER entry.count > 0
       SORT entry.count DESC
       LIMIT 10
       RETURN {requirement: entry.requirement, satisfier_count: entry.count}
   )
   
   RETURN {
     top_ten_satisfied: top_ten_satisfied,
     unsatisfied_count: L

## 3. The graph can be asked how it was built

Every relation carries `stated`: true if the lexer read it out of the syntax, absent
if the LLM inferred it. So "how much of this graph is read and how much is guessed"
is itself a query -- per model, in one pass.

In [4]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

AQL
   WITH sysml_Entities, sysml_Relations
   
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO"
     LET from = DOCUMENT(r._from)
     FILTER from != null
     FOR model IN from.models
       COLLECT m = model, read = r.stated == true WITH COUNT INTO n
       RETURN {model: m, source: read ? "read from the syntax" : "inferred by the LLM", relations: n}

rows (6, first 6)
   {"model": "apollo-11-sysml-v2", "source": "inferred by the LLM", "relations": 904}
   {"model": "apollo-11-sysml-v2", "source": "read from the syntax", "relations": 2938}
   {"model": "Drone_BaseArchitecture", "source": "inferred by the LLM", "relations": 7}
   {"model": "Drone_BaseArchitecture", "source": "read from the syntax", "relations": 8}
   {"model": "DroneModelLogical", "source": "inferred by the LLM", "relations": 23}
   {"model": "DroneModelLogical", "source": "read from the syntax", 

The read majority is the point of the second pass. What the LLM contributes is the
1,000-odd relations that are in the *prose* -- `refines`, `dependson`, `performs` --
which no lexer can find.

## 4. An element has two names, and both work

`requirement def <'HLR-R001'> CrewReturnSafetyRequirement` can be addressed either
way, and an engineer will use the identifier. Extraction, reading prose, kept
whichever name the sentence it was reading happened to use -- so requirements
arrived twice, once under each name, with the edges split between the two halves.
The lexer reads the declaration, which is the only place that says they are one
thing, and folds them together. Asking by the identifier now reaches the whole
element.

In [5]:
nl.instance().ask(
    "Trace the requirement whose short name is HLR-R001: what satisfies it, what "
    "refines it, and what it is related to in either direction."
).show(row_limit=8)

Q  Trace the requirement whose short name is HLR-R001: what satisfies it, what refines it, and what it is related to in either direction.

AQL
   WITH sysml_Entities, sysml_Relations
   LET requirement = FIRST(
     FOR e IN sysml_Entities
       FILTER e.short_name == "HLR-R001"
       RETURN e
   )
   FOR v, r IN 1..1 ANY requirement sysml_Relations
     FILTER r.type == "RELATED_TO" AND r.relationship_type IN ["satisfies", "refines", "related"]
     RETURN {
       relation: r.relationship_type,
       related_entity: v.entity_name,
       direction: r._from == requirement._id ? "outgoing" : "incoming"
     }

rows (17, first 8)
   {"relation": "refines", "related_entity": "FLR-R015", "direction": "incoming"}
   {"relation": "refines", "related_entity": "CONTINGENCYPLANNINGANDRAPIDRESPONSE", "direction": "incoming"}
   {"relation": "refines", "related_entity": "DEEPSPACEHABITATIONANDLIFESUPPORT", "direction": "incoming"}
   {"relation": "refines", "related_entity": "ATMOSPHERICREENT

## 5. Joining a layer that is not in any file

The `SIMILAR_TO` edges are computed, not declared -- autograph's `SimilarityFinder`
matching entities of the same kind across model boundaries. They are queryable like
anything else, so "what does the drone have in common with Apollo" is a join.

In [6]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

Q  Which requirements does the drone model state that the Apollo model has an analogous requirement for, and how close are they?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR droneReq IN sysml_Entities
     FILTER 'DroneModelLogical' IN droneReq.models
     FILTER droneReq.entity_type == 'requirement'
     FOR apolloReq, similarityEdge IN 1..1 ANY droneReq sysml_Relations
       FILTER similarityEdge.type == 'SIMILAR_TO'
       FILTER 'apollo-11-sysml-v2' IN apolloReq.models
       FILTER apolloReq.entity_type == 'requirement'
       RETURN {
         drone_requirement: droneReq.entity_name,
         apollo_analogous_requirement: apolloReq.entity_name,
         similarity_cosine: similarityEdge.cosine,
         similarity_description: similarityEdge.description
       }

rows (25, first 6)
   {"drone_requirement": "DE-REQ-9 SAFETY", "apollo_analogous_requirement": "SHN-N029", "similarity_cosine": 0.5528614970677261, "similarity_description": "DE-REQ-9 SAFETY in the DroneModelLogi

### Where AQLizer stops

It needs the question to land on a field. Ask it something whose answer is spread
through the `doc` comments of a dozen requirements in four files and there is no
column to filter on -- which is the next section.

---

# Part 2 -- GraphRAG

Three scopes, all upstream, all reading this graph.

  `local`    hybrid vector + BM25 over the entities, fused, then expanded over the
             relations it lands on
  `unified`  the source chunks and the entity graph searched in parallel
  `global`   the community reports, map-reduced

## 6. `local` -- a question in words the model never uses

No SysML file contains "alive", "breathing" or "keeps". The elements are called
`PLSS`, `PSA`, `EnvironmentalControlSystem`. Vector search does not care.

In [7]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show(row_limit=4)

Q  What keeps the astronauts alive and breathing, and what limits does it have to hold?

retrieved  28 documents, 48 edges, 73,795 chars of context

cited (14, first 4)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 10, "source": "models/apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml"}
   {"cite": 11, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 12, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

### Life Support and Oxygen Supply

The astronauts' survival and ability to breathe in space are reliant on the life support systems, specifically the Portable Life Support System (PLSS). This system is responsible for providing breathable oxygen, hence maintaining the astronaut's life support during extravehicular activities (EVAs), and must operate efficiently to manage the internal environment, inclu

## 7. `unified` -- a figure that never became an entity

Some numbers live only in a `doc` comment, so they are in the source text and in no
`attributes` map. `unified` searches the chunks and the graph together, which is what
reaches them.

The second call is the important one: `evidence(find=...)` prints the retrieved text
around the figure, so the answer can be checked against what was actually read rather
than taken on trust.

In [8]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

Q  How much drinking water must the environmental control system supply per crew member per day?

retrieved  10 documents, 81 edges, 12,940 chars of context

cited (3, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

The models indicate that the environmental control system is required to supply a minimum of 2 kilograms of potable water per crew member per day[CITE:1].

evidence  (420 chars at char 2,689, of 12,940 retrieved)
   		}
   	}
   	requirement def <'CLR-R064'> ECSWaterSupplyRate {
   		doc /* The Apollo 11 Mission's crew's potable water system shall provide a minimum of 2 kilograms of potable water per crew member per day. */
   		@Rationale {
   			text = "Adequate potable water is a fundamental r

The number in the answer is in the `doc` comment printed underneath it, and the
citation resolves to the file it came from. That is the difference between a
retrieval and a recollection.

In [9]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

Q  What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?

retrieved  12 documents, 121 edges, 11,905 chars of context

cited (5, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Analysis/AnalysisPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

The minimum delta-v that the Lunar Module Ascent Stage (LMAS) must provide is 1,850 m/s. This requirement is critical for the LMAS to achieve lunar orbit post-surface operations[CITE:1][CITE:3]. The rationale behind this figure is to specify the necessary propulsion performance for escaping the Moon's gravitational pull and ensuring successful insertion into lunar orbit[CITE:1].

### Sources Cited
- **CLR-R115** defines this specific requirement for the Delta V provided by the LM ascent stage [CITE:1][CITE:3]. 


## 8. `global` -- a question no single row answers

`global` never looks at an entity. It reads the 138 community reports the extraction
step wrote, scores them against the question, and summarises the ones that survive --
so it answers about the corpus as a whole.

In [10]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

Q  What concerns are these models organised around, and what does each part of the corpus contribute?

retrieved  27 community reports -> 11 points

A  # Concerns and Contributions of the Apollo11Model

The Apollo11Model is a systematically organized framework designed to support various mission operations relevant to the Apollo missions. Below, we highlight the key areas of concern the models address and the specific contributions of different packages and modules.

## Core Organisation and Integration

The **Apollo11Model** serves as the central framework, integrating various specialized packages such as the **MissionPackage**, **CoSMAPackage**, and **CapabilitiesPackage**. These packages provide a structure for mission-related data and functionalities, crucial for executing Apollo missions effectively.

## Command/Service Module

A pivotal component within these models is the **Command/Service Module**, which plays a critical role in satisfying mission-critical requirements. It prov

---

# The line between them

One question, both engines.

In [11]:
QUESTION = "How many Apollo requirements does nothing satisfy?"

nl.instance().ask(QUESTION).show(row_limit=2)

Q  How many Apollo requirements does nothing satisfy?

AQL
   LET unsatisfied = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       FILTER LENGTH(
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
           RETURN 1
       ) == 0
       RETURN e.entity_name
   )
   RETURN {total: LENGTH(unsatisfied), examples: SLICE(unsatisfied, 0, 10)}

rows (1, first 1)
   {"total": 396, "examples": ["SHN-N021: INVESTMENTJUSTIFICATION", "REQUIREMENT CLR-R069", "FUNCTIONALREQUIREMENTSPACKAGE: FLR-R066", "SHN-N021", "CREW EMERGENCY MEDICAL PROFICIENCY", "LOW EARTH ORBIT DELTA V", "SHN-N019: NATIONALPRESTIGE", "COMMUNICATIONS", "SHN-N014: CONTINUOUSSUPPORT", "SHN-N003"]}

A  There are a total of 396 Apollo requirements that remain unsatisfied. Some examples of these unsatisfied requirements include "SHN-N021: INVESTMENTJUSTIFICATION," "REQUIREMENT CLR-R069," "FUNCTIONALREQUIREMENT

In [12]:
(await nl.retriever().ask_async(QUESTION)).show(row_limit=3)

Q  How many Apollo requirements does nothing satisfy?

retrieved  24 documents, 48 edges, 43,504 chars of context

cited (6, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/CoSMA/CoSMAViewsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Purpose/MissionSpecificationPackage.sysml"}

A  The context provided does not clearly state the exact number of Apollo requirements that nothing satisfies. The model does not directly address this information. Therefore, the model does not say how many Apollo requirements nothing satisfies.



AQLizer counts them. GraphRAG cannot: it retrieves requirements that *look* relevant
and describes them, because there is no passage anywhere that states how many
requirements lack a relation. Reverse the question -- "what keeps the astronauts
alive" -- and AQLizer has nothing to filter on while GraphRAG answers from four files.

So the rule is about the question, not the engine:

| the question is about | use |
|---|---|
| a number, a count, a ranking, a rollup | AQLizer |
| something absent -- unsatisfied, unowned, uncovered | AQLizer |
| provenance, or the shape of the graph itself | AQLizer |
| a concept the model spells differently | GraphRAG `local` |
| a figure written in prose rather than declared | GraphRAG `unified` |
| the corpus as a whole | GraphRAG `global` |

Both are pointed at a graph the importer's own writer produced, and neither has a
hand-written query behind it. When an answer is wrong, the fix goes in
`sysml/aql_examples.md` -- two of the queries above are only correct because a
previous wrong answer was turned into a worked example there.